## Reproducing QG configuration dataset 

This notebook can be used to produce the QG datasets for both the reference (DNS) and the emulator (based on the coarse-resolution solver). The construction of the dataset sub-trajectories $\mathrm{dim}(\mathbb{D})$ is based on fractions of the chaotic decorrelation of the system and the turnover time $t_L$.

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import tqdm
import h5py
import matplotlib.pyplot as plt

plt.rcParams.update({
    'mathtext.fontset': 'cm'
})

import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jnr

jax.config.update(
    'jax_enable_x64', True
)

import models.time_solver as stepper
from models.qg_periodic import (
    QgPeriodic, 
    dynamical_solver,
    subgrid_fluxes,
    enstrophy,
    turnover_time,
    iso_spectrum,
)
from utils import (
    into_s,
    from_s,
    spectral_pad,
)

### Loading from a configuration snapshot

We load a pre-existing snapshot, previously generated using the `qg-spinup.py` script.

In [ ]:
cfg_name = 'qg-det'
data_path = os.path.join(os.path.join(os.getcwd(), '../data'), cfg_name)

key = jnr.key(42)
eq, t0, om_s = QgPeriodic.load(os.path.join(data_path, 'spinup.h5'))
print(eq)

dt = 2e-4
sigma = 10.0
k_f = 15

iso_K = np.sqrt(eq.lap)
forcing_spectrum = np.full_like(iso_K, sigma)
forcing_spectrum[iso_K < k_f - 1] = 0
forcing_spectrum[iso_K > k_f + 1] = 0
forcing_spectrum[iso_K == 0] = 0

forcing_det = into_s(np.cos(k_f * eq.X) + np.cos(k_f * eq.Y))

def source(om_s, t):
    f_s = forcing_spectrum * forcing_det
    return f_s

### Instantiating a new dynamical solver

To accumulate statistics, we need to run new iterations at steady state. We create a new dynamical solver with the same configuration parameters and run for $1/100$ of the spinup time, and compute turnover times $t_L(t)$ before averaging.

In [ ]:
solver = jax.jit(dynamical_solver(
    eq,
    stepper.BPR353(dt),
    source
))

# 1/50 of spinup
iters = int((t0 / 50) / dt)
logs = 1000
logs_freq = int(iters / logs)

time_t = []
eke_t = []
ens_t = []
turnover_t = []
ux_p_t = []
ek_k_t = []

time = t0
pbar = tqdm.tqdm(range(iters), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for i in pbar:
    c, om_s = solver(om_s, time)
    time += dt
    if i % logs_freq == 0:
        time_t.append(time)

        turnover_t.append(
            turnover_time(om_s)
        )

        pbar.set_postfix(
            cfl=format(dt / c, ".2f")
        )

In [ ]:
print('turnover time =',np.mean(turnover_t, axis=0))

### Computing decorrelation time

To estimate the decorrelation time, we measure the vorticity root mean squared error between a reference trajectory $\omega(t)$ and a perturbed one $\delta \omega(t)$. This process is repeated for an ensemble.

*Note*: the perburted trajectories are initialized with a small linear perturbation from a randomly shifted gaussian at amplitude $10^{-10}$.

In [ ]:
fig, axs = plt.subplots(ncols=1, nrows=1, figsize=(2.5, 3.5), dpi=120)

_, _, om_s = QgPeriodic.load(os.path.join(data_path, 'spinup.h5'))
gaussian = into_s(np.exp(-10 * np.pi * ((eq.X - np.pi)**2 + (eq.Y - np.pi)**2)))

key, pert_key = jnr.split(key)
s_x, s_y = jax.random.uniform(pert_key, shape=(2,), minval=-np.pi, maxval=np.pi)

om_s_p = om_s + 1e-10 * jnp.exp(-1j * (eq.k_x * s_x + eq.k_y * s_y)) * gaussian

solver = jax.jit(dynamical_solver(
    eq,
    stepper.BPR353(dt),
    source
))

T = 30
iters = int(T / dt)
logs = 1000
logs_freq = int(iters / logs)

time_t = []
rmse_t = []

time = 0.0
pbar = tqdm.tqdm(range(iters), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for i in pbar:
    c, om_s = solver(om_s, time)
    _, om_s_p = solver(om_s_p, time)
    time += dt
    if i % logs_freq == 0:
        time_t.append(time)

        rmse_t.append(
            np.sqrt(2 * enstrophy(om_s_p - om_s))
        )

        pbar.set_postfix(
            cfl=format(dt / c, ".2f"),
            rms=rmse_t[-1],
        )

axs.semilogy(
    time_t, 
    rmse_t, 
    color='k'
)

decorr_t = 25
axs.axvline(decorr_t, linestyle='--', color='k')

axs.set_xlabel(r'$t$', fontsize=15)
axs.set_ylabel(r'$\langle (\delta \omega(t) - \omega(t))^{2} \rangle^{1/2}$', fontsize=15)
axs.tick_params(reset=True, axis='both', which='both', direction='in')

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'decorr.pdf'))
plt.show()

In [ ]:
def get_dataset_stats(
    ratio: int, 
    dt: float,
    decorr_t: float,
    frac_decorr: float,
    turnover_t: float,
    total_turnovers: float,
):
    n_decorr_iters = int(decorr_t / dt)
    print('Iterations required for full decorrelation =',n_decorr_iters)

    n_steps = int(frac_decorr * n_decorr_iters / ratio)
    print('Number of steps in sub-trajectories =',n_steps)

    dataset_timespan = turnover_t * total_turnovers
    print('Total dataset timespan =',dataset_timespan)

    n_trajs = int(dataset_timespan / (dt * ratio) / n_steps)
    print('Number of sub-trajectories =',n_trajs)

    n_samples = n_trajs * n_steps
    print('Dataset samples =',n_samples)
    
    return (
        n_steps, n_trajs
    )

ratio = 16
n_steps, n_trajs = get_dataset_stats(
    ratio=ratio,
    dt=dt,
    decorr_t=decorr_t, # Full decorrelation at t ~ 25
    frac_decorr=0.003,
    turnover_t=0.81,
    total_turnovers=1
)

### Visualizing the effect of coarse-graining:

In [ ]:
ps_s = -eq.ilap * om_s
om_g = from_s(om_s)

eq_coarse = QgPeriodic(
    nu=eq.nu,
    mu=eq.mu,
    beta=eq.beta,
    n_kx=int((eq.n_kx - 1) / ratio + 1),
    n_ky=int(eq.n_ky / ratio)
)

print(eq_coarse)

om_s_coarse = spectral_pad(om_s, eq_coarse.n_kx, eq_coarse.n_ky)
ps_coarse_s = -eq_coarse.ilap * om_s_coarse
om_g_coarse = from_s(om_s_coarse)

fig, axs = plt.subplots(ncols=3, nrows=1, figsize=(11.0, 3.5), width_ratios=[1, 1, 0.8])
contour = axs[0].contourf(eq.X, eq.Y, om_g, cmap='twilight', levels=45, vmin=-40, vmax=40)
fig.colorbar(contour, ax=axs[0])

axs[0].set_xlabel(r'$x$', fontsize=15)
axs[0].set_ylabel(r'$y$', fontsize=15)

contour = axs[1].contourf(eq_coarse.X, eq_coarse.Y, om_g_coarse, cmap='twilight', levels=45, vmin=-40, vmax=40)
fig.colorbar(contour, ax=axs[1])

axs[1].set_xlabel(r'$x$', fontsize=15)
axs[1].set_ylabel(r'$y$', fontsize=15)

kr, bin_max, ek_k = iso_spectrum(eq, eq.lap * np.abs(ps_s)**2)
axs[2].loglog(kr[:bin_max], ek_k[:bin_max], color='k')
kr_coarse, bin_max_coarse, ek_k_coarse = iso_spectrum(eq_coarse, eq_coarse.lap * np.abs(ps_coarse_s)**2)
axs[2].loglog(kr_coarse[:bin_max_coarse], ek_k_coarse[:bin_max_coarse], linestyle='--')

axs[2].axvline(15 - 0.5, color='lightgray', linestyle='--', label=r'$k_f$')

axs[2].set_xlabel(r'$E(k)$', fontsize=15)
axs[2].set_ylabel(r'$k$', fontsize=15)
axs[2].tick_params(reset=True, axis='both', which='both', direction='in')
axs[2].legend(fontsize=13, frameon=False)

fig.tight_layout()
plt.show()

### Generating dataset from the DNS (for the SGS learning; step 2):

In [ ]:
# reset
eq, t0, om_s = QgPeriodic.load(
    os.path.join(data_path, 'spinup.h5')
)

solver = jax.jit(dynamical_solver(
    eq,
    stepper.BPR353(dt),
    source
))

iters = n_trajs * n_steps * ratio

time_t = []
om_c_t = []
tau_x_t = []
tau_y_t = []

time = t0
pbar = tqdm.tqdm(range(iters), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for i in pbar:
    c, om_s = solver(om_s, time)
    time += dt

    if i % ratio == 0:
        time_t.append(time)
        
        om_c_t.append(
            spectral_pad(om_s, eq_coarse.n_kx, eq_coarse.n_ky)
        )

        tau_x, tau_y = subgrid_fluxes(eq, eq_coarse, om_s)
        tau_x_t.append(
            tau_x
        )
        tau_y_t.append(
            tau_y
        )
    
om_c_ref = np.array(np.split(np.array(om_c_t), n_trajs, axis=0))
time_ref = np.array(np.split(np.array(time_t), n_trajs, axis=0))
tau_x_ref = np.array(np.split(np.array(tau_x_t), n_trajs, axis=0))
tau_y_ref = np.array(np.split(np.array(tau_y_t), n_trajs, axis=0))
print('Training reference (DNS) dataset shape =',om_c_ref.shape)

### Generating dataset from the coarse-resolution solver (for the emulator learning; step 1):

In [ ]:
dt_coarse = dt * ratio

iso_K_coarse = np.sqrt(eq_coarse.lap)
forcing_spectrum_coarse = np.full_like(iso_K_coarse, sigma)
forcing_spectrum_coarse[iso_K_coarse < k_f - 1] = 0
forcing_spectrum_coarse[iso_K_coarse > k_f + 1] = 0
forcing_spectrum_coarse[iso_K_coarse == 0] = 0

forcing_det_coarse = into_s(np.cos(k_f * eq_coarse.X) + np.cos(k_f * eq_coarse.Y))

def source_coarse(om_g, t):
    f_s = forcing_spectrum_coarse * forcing_det_coarse
    return f_s

# Baseline "coarse" without correction
solver_coarse = jax.jit(dynamical_solver(
    eq_coarse,
    stepper.BPR353(dt_coarse),
    source_coarse
))

om_c_emu = np.zeros_like(om_c_ref)
pbar = tqdm.tqdm(range(n_trajs), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for traj in pbar:
    om_c_emu[traj, 0] = om_c_ref[traj, 0]
    time = time_ref[traj, 0]
    for step in range(1, n_steps):
        _, om_c_emu[traj, step] = solver_coarse(om_c_emu[traj, step - 1], time)
        time += dt_coarse

In [ ]:
print('Saving datasets...')
with h5py.File(os.path.join(data_path, 'datasets.h5'), 'w') as f:
    f.attrs['dt'] = dt
    f.attrs['ratio'] = ratio
    f.attrs['n_steps'] = n_steps
    f.attrs['n_trajs'] = n_trajs
    
    f.attrs['sigma'] = sigma
    f.attrs['k_f'] = k_f

    f.create_dataset('om_c_ref',
                     data=np.array(om_c_ref))
    f.create_dataset('tau_x_ref',
                     data=np.array(tau_x_ref))
    f.create_dataset('tau_y_ref',
                     data=np.array(tau_y_ref))
    f.create_dataset('om_c_emu',
                     data=np.array(om_c_emu))
eq.save(
    os.path.join(data_path, 'snapshot.h5'),
    t0 + iters * dt,
    om_s
)